# Integrate RocketRide into an existing app

**Time:** 15 minutes

**What you build:** TODO(joshua): one or two sentences.

**Done when:** TODO(joshua): the visible result that means you're finished.

You'll clone a small TypeScript notepad into `./notepad-starter`, wire a RocketRide pipeline into it with the TypeScript SDK, and run it locally.

> Run the cells from top to bottom with **Shift+Enter**. Every cell is safe to run again.
> New to notebooks or haven't set up yet? Start with [SETUP.md](https://github.com/rocketride-org/rocketride-workshops/blob/main/SETUP.md).

## Preflight

Checks Python, the `rocketride` package, your API key and the connection to RocketRide Cloud, plus Node, pnpm and git. Fix anything marked ❌, then run it again.

In [ ]:
# Preflight: checks your machine. Prints one green "ready" line when everything passes.
import re, shutil, subprocess, sys, urllib.error, urllib.request

problems, warnings = [], []

if sys.version_info < (3, 10):
    problems.append(f"Python {sys.version.split()[0]} is too old. Fix: install Python 3.10+ "
                    "and select it as this notebook's kernel (see SETUP.md).")

try:
    from rocketride import RocketRideClient
    if not RocketRideClient().get_apikey():
        warnings.append("No RocketRide API key in your environment or .env yet. "
                        "The Config cell below will ask for it.")
except ImportError:
    problems.append("The rocketride package is missing. Fix: run  %pip install rocketride  "
                    "in a new cell, then restart the kernel.")

try:
    urllib.request.urlopen("https://api.rocketride.ai", timeout=10)
except urllib.error.HTTPError:
    pass  # the server answered, so it's reachable
except Exception as e:
    problems.append(f"Can't reach api.rocketride.ai ({e}). Fix: check your internet "
                    "connection, VPN or proxy, then run this cell again.")

def tool_version(name):
    path = shutil.which(name)
    if not path:
        return None
    out = subprocess.run([path, "--version"], capture_output=True, text=True, timeout=30).stdout
    m = re.search(r"(\d+)\.(\d+)", out)
    return (int(m[1]), int(m[2])) if m else (0, 0)

for name, need, fix in [
    ("node", (20, 0), "install Node.js 20 or newer from nodejs.org"),
    ("pnpm", (10, 0), "run  npm install -g pnpm@latest"),
    ("git", (0, 0), "install git from git-scm.com"),
]:
    have = tool_version(name)
    if have is None:
        problems.append(f"{name} is not installed or not on your PATH. Fix: {fix}, "
                        "then restart VS Code.")
    elif have < need:
        problems.append(f"{name} {have[0]}.{have[1]} is too old (need {need[0]}+). Fix: {fix}.")

for w in warnings:
    print("⚠️ ", w)
if problems:
    for p in problems:
        print("❌", p)
    raise RuntimeError(f"Preflight failed: {len(problems)} problem(s). Fix them, then run this cell again.")
print("\033[32m✅ ready\033[0m")

## Config

Reads your RocketRide settings from the environment or a `.env` file next to this notebook. If your API key isn't set, it asks for it. Nothing is saved.

In [ ]:
# Config: the ONLY cell that reads environment variables. Every later cell uses CONFIG.
import getpass, os
from pathlib import Path

def read_dotenv(path=Path(".env")):
    values = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            key, sep, value = line.strip().partition("=")
            if sep and key.strip().startswith("ROCKETRIDE_"):
                values[key.strip()] = value.strip().strip("\"'")
    return values

env = read_dotenv()  # a .env file next to this notebook...
env.update({k: v for k, v in os.environ.items() if k.startswith("ROCKETRIDE_")})  # ...env wins
env.setdefault("ROCKETRIDE_URI", "https://api.rocketride.ai")
if not env.get("ROCKETRIDE_APIKEY"):
    previous = globals().get("CONFIG", {}).get("apikey")  # don't re-prompt on re-run
    env["ROCKETRIDE_APIKEY"] = previous or getpass.getpass("RocketRide API key: ").strip()

CONFIG = {
    "uri": env["ROCKETRIDE_URI"],
    "apikey": env["ROCKETRIDE_APIKEY"],
    "deploy_uri": env.get("ROCKETRIDE_DEPLOY_URI"),
    "deploy_apikey": env.get("ROCKETRIDE_DEPLOY_APIKEY"),
    "env": env,
}
print("Server: ", CONFIG["uri"])
print("API key:", "set" if CONFIG["apikey"] else "MISSING")
print("Deploy: ", CONFIG["deploy_uri"] or "not configured")

In [ ]:
# Connect: one shared RocketRide client for every cell below. Safe to re-run.
from rocketride import RocketRideClient

if "client" not in globals() or not client.is_connected():
    client = RocketRideClient(uri=CONFIG["uri"], auth=CONFIG["apikey"], env=CONFIG["env"])
    await client.connect()
print("Connected to", CONFIG["uri"])

async def start(pipe_path):
    """Start a pipeline, or attach to it if it's already running. Returns the task token."""
    result = await client.use(filepath=str(pipe_path), use_existing=True)
    return result["token"]

## Stage 1: Get the notepad running (3 min)

**Goal:** clone the notepad app, install its dependencies and start it at http://127.0.0.1:5173.

**Watch for:** the install takes a minute on slow wifi. If port 5173 is busy, see the facilitator.

In [ ]:
# Clone the notepad app into ./notepad-starter (skips if it's already there)
import shutil, subprocess
from pathlib import Path

NOTEPAD = Path("notepad-starter")
NOTEPAD_REPO = "https://github.com/rocketride-org/notepad-starter"

def sh(*args, cwd=None):
    """Run a command and show its output. Raises if it fails."""
    r = subprocess.run([shutil.which(args[0]) or args[0], *args[1:]], cwd=cwd,
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    print((r.stdout + r.stderr).strip()[-2000:])
    r.check_returncode()

if (NOTEPAD / ".git").exists():
    print("Already cloned:", NOTEPAD.resolve())
else:
    sh("git", "clone", NOTEPAD_REPO, str(NOTEPAD))

In [ ]:
# Install the app's dependencies (safe to re-run)
sh("pnpm", "install", cwd=NOTEPAD)

In [ ]:
# Start the dev server in the background (skips if it's already running)
import socket, time

def port_open(port=5173):
    with socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0

if not port_open():
    NOTEPAD_DEV = subprocess.Popen(
        [shutil.which("pnpm"), "dev", "--host", "127.0.0.1", "--port", "5173", "--strictPort"],
        cwd=NOTEPAD, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if port_open():
            break
        time.sleep(0.5)
print("Notepad:", "http://127.0.0.1:5173" if port_open() else "did not start, see FACILITATOR.md")

In [ ]:
# Skip ahead: run this only if the stage above didn't work
# Resets the app to the plain starter, then starts it again with the cell above.
sh("git", "fetch", "origin", "main", cwd=NOTEPAD)
sh("git", "checkout", "origin/main", "--", ".", cwd=NOTEPAD)
sh("pnpm", "install", cwd=NOTEPAD)

## Stage 2: Add the RocketRide client (5 min)

**Goal:** add the RocketRide TypeScript SDK and one shared, connected client.

**Watch for:** TODO(joshua)

In [ ]:
# Add the RocketRide TypeScript SDK (safe to re-run)
sh("pnpm", "add", "rocketride", cwd=NOTEPAD)

In [ ]:
%%writefile notepad-starter/src/rocketride.ts
// TODO(joshua): the shared RocketRide client (see the integrated branch for the finished file).
export {};

In [ ]:
# Skip ahead: run this only if the stage above didn't work
FILES = ["package.json", "pnpm-lock.yaml", "src/rocketride.ts", "src/vite-env.d.ts"]  # TODO(joshua): confirm
sh("git", "fetch", "origin", "integrated", cwd=NOTEPAD)
sh("git", "checkout", "origin/integrated", "--", *FILES, cwd=NOTEPAD)
sh("pnpm", "install", cwd=NOTEPAD)

## Stage 3: Call your pipeline from the app (5 min)

**Goal:** TODO(joshua): the pipeline-backed feature.

**Watch for:** TODO(joshua)

In [ ]:
%%writefile notepad-starter/src/assist.ts
// TODO(joshua): the pipeline-backed feature and its call site.
export {};

In [ ]:
# Run it: the dev server reloads on save. Open the app and try your feature.
print("Open http://127.0.0.1:5173" if port_open() else "Start the dev server with the Stage 1 cell first.")

In [ ]:
# Skip ahead: run this only if the stage above didn't work
FILES = ["src/main.ts", "src/style.css", "src/rocketride.ts"]  # TODO(joshua): confirm
sh("git", "fetch", "origin", "integrated", cwd=NOTEPAD)
sh("git", "checkout", "origin/integrated", "--", *FILES, cwd=NOTEPAD)

## Clean up

Stops the dev server this notebook started.

In [ ]:
# Stop the dev server (safe to re-run)
import os

dev = globals().get("NOTEPAD_DEV")
if dev and dev.poll() is None:
    if os.name == "nt":
        subprocess.run(["taskkill", "/T", "/F", "/PID", str(dev.pid)], capture_output=True)
    else:
        dev.terminate()
    print("Stopped the dev server.")
else:
    print("No dev server started from this notebook.")

## What's next

- **Docs:** [docs.rocketride.org](https://docs.rocketride.org)
- **Help and community:** [RocketRide Discord](https://discord.gg/PMXrtenMsY)
- **Next module:** [Build and deploy an app](https://github.com/rocketride-org/rocketride-workshops/blob/main/modules/app-build/app-build.ipynb)